# FLUX Dynamic Model Inspector

**Dynamic inspection of Flux-Apex-V1.flx without hardcoded expectations.**

This notebook discovers the actual structure of the .flx file at runtime,
supporting all versions: v4.0, v5.0-voice, v6.0-autonomous, v7.x-fabric.

**Features:**
- Schema-free component discovery
- Native FLUX component analysis
- Embedded model detection (language, vision, audio)
- Detection model analysis (face, pose, depth, object)
- Parameter counts and memory estimation
- Lazy loading support checking

---

In [ ]:
"""Cell 1: Environment Setup"""
import os
import sys
from pathlib import Path

# Detect environment
IN_KAGGLE = os.path.exists('/kaggle')
IN_COLAB = 'google.colab' in sys.modules

if IN_KAGGLE:
    ROOT = Path('/kaggle/working/FLUX')
    if not ROOT.exists():
        !git clone https://github.com/Unseengap/FLUX.git {ROOT}
    os.chdir(ROOT)
elif IN_COLAB:
    ROOT = Path('/content/FLUX')
    if not ROOT.exists():
        !git clone https://github.com/Unseengap/FLUX.git {ROOT}
    os.chdir(ROOT)
else:
    ROOT = Path('.').resolve()
    while ROOT.name and not (ROOT / 'flux_utils.py').exists():
        ROOT = ROOT.parent
    if not (ROOT / 'flux_utils.py').exists():
        ROOT = Path('/Users/admin/Desktop/flux')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")
print(f"Root: {ROOT}")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

print(f"\u2713 Environment configured")

In [ ]:
"""Cell 2: Device & Token Setup"""
import torch
from flux_utils import _resolve_hf_token

# Device setup
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# HuggingFace token
HF_TOKEN = None
if IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
        if HF_TOKEN: print("\u2713 HF_TOKEN from Kaggle secrets")
    except: pass
elif IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        if HF_TOKEN: print("\u2713 HF_TOKEN from Colab secrets")
    except: pass
if not HF_TOKEN:
    HF_TOKEN = _resolve_hf_token()
    if HF_TOKEN: print("\u2713 HF_TOKEN from environment")
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
else:
    print("\u26a0 No HF_TOKEN found - may need for download")

In [ ]:
"""Cell 3: Download/Locate Flux-Apex-V1.flx"""
from huggingface_hub import hf_hub_download

HF_REPO = "UnseenGAP/FLUX"
CHECKPOINTS_DIR = ROOT / 'checkpoints'
CHECKPOINTS_DIR.mkdir(exist_ok=True)

MODEL_PATH = CHECKPOINTS_DIR / 'Flux-Apex-V1.flx'

if MODEL_PATH.exists():
    print(f"\u2713 Local model found: {MODEL_PATH}")
    print(f"  Size: {MODEL_PATH.stat().st_size / 1e9:.2f} GB")
else:
    print("Downloading Flux-Apex-V1.flx from HuggingFace...")
    try:
        hf_hub_download(
            repo_id=HF_REPO,
            filename='checkpoints/Flux-Apex-V1.flx',
            local_dir=str(ROOT),
            token=HF_TOKEN,
        )
        print(f"\u2713 Downloaded: {MODEL_PATH}")
    except Exception as e:
        print(f"\u2717 Download failed: {e}")
        print("\nPlease download manually or set HF_TOKEN")

In [ ]:
"""Cell 4: Dynamic Structure Inspection"""
from flux_inspector import inspect_flx, generate_report, get_component_tree

print("Inspecting Flux-Apex-V1.flx structure...")
print("=" * 60)

# Full inspection
report = inspect_flx(MODEL_PATH)

# Print summary
print(generate_report(report, format='text'))

In [ ]:
"""Cell 5: Parameter Breakdown by Category"""
print("Parameter Breakdown")
print("=" * 60)

# Native FLUX
native_params = sum(c.parameters for c in report.native_components.values())
print(f"\nNative FLUX Components: {native_params:,} params")
for name, info in sorted(report.native_components.items(), key=lambda x: -x[1].parameters)[:10]:
    pct = info.parameters / report.total_parameters * 100 if report.total_parameters > 0 else 0
    print(f"  {name:20}: {info.parameters:>15,} ({pct:>5.1f}%)")

# Embedded models
model_params = sum(c.parameters for c in report.embedded_models.values())
print(f"\nEmbedded Models: {model_params:,} params")
for name, info in sorted(report.embedded_models.items(), key=lambda x: -x[1].parameters):
    pct = info.parameters / report.total_parameters * 100 if report.total_parameters > 0 else 0
    base = info.base_model[:35] + '...' if info.base_model and len(info.base_model) > 35 else (info.base_model or 'N/A')
    print(f"  {name:15}: {info.parameters:>12,} ({pct:>5.1f}%) - {base}")

# Detection models
detection_params = sum(c.parameters for c in report.detection_models.values())
print(f"\nDetection Models: {detection_params:,} params")
for name, info in sorted(report.detection_models.items(), key=lambda x: -x[1].parameters):
    pct = info.parameters / report.total_parameters * 100 if report.total_parameters > 0 else 0
    print(f"  {name:15}: {info.parameters:>12,} ({pct:>5.1f}%) - {info.component_type}")

print(f"\n{'='*60}")
print(f"TOTAL: {report.total_parameters:,} parameters")

In [ ]:
"""Cell 6: Component Availability Matrix"""
print("Component Availability Matrix")
print("=" * 60)

print("\n{:25} {:^10} {:^12} {:^10}".format('Component', 'Enabled', 'Has Weights', 'Lazy Load'))
print("-" * 60)

# Native components
print("\n[Native FLUX]")
for name, info in report.native_components.items():
    enabled = report.component_flags.get(name, True)
    enabled_str = '\u2713' if enabled else '\u2717'
    weights_str = '\u2713' if info.has_weights else '\u2717'
    lazy_str = '-'
    legacy = ' [LEGACY]' if info.legacy else ''
    print(f"  {name:23} {enabled_str:^10} {weights_str:^12} {lazy_str:^10}{legacy}")

# Embedded models
if report.embedded_models:
    print("\n[Embedded Models]")
    for name, info in report.embedded_models.items():
        enabled_str = '\u2713'
        weights_str = '\u2713' if info.has_weights else '\u2717'
        lazy_str = 'Yes' if info.lazy_load else 'No'
        print(f"  {name:23} {enabled_str:^10} {weights_str:^12} {lazy_str:^10}")

# Detection models
if report.detection_models:
    print("\n[Detection Models]")
    for name, info in report.detection_models.items():
        enabled_str = '\u2713'
        weights_str = '\u2713' if info.has_weights else '\u2717'
        lazy_str = 'Yes' if info.lazy_load else 'No'
        onnx = ' [ONNX]' if info.component_type == 'onnx' else ''
        print(f"  {name:23} {enabled_str:^10} {weights_str:^12} {lazy_str:^10}{onnx}")

In [ ]:
"""Cell 7: Test Native Components via FLUXModel"""
from flux_model import FLUXModel

print("Testing FLUXModel API")
print("=" * 60)

# Load via FLUXModel (recommended API)
model = FLUXModel.load(MODEL_PATH)

# Print summary using the new method
print("\nModel Summary via FLUXModel.summary():")
model.summary()

# Test get_component
print("\nTesting get_component():")
for comp in ['cse', 'field', 'memory', 'bridges']:
    data = model.get_component(comp)
    if data:
        keys = list(data.keys()) if isinstance(data, dict) else 'N/A'
        print(f"  \u2713 {comp}: {len(keys) if isinstance(keys, list) else keys} keys")
    else:
        print(f"  \u2717 {comp}: not found")

In [ ]:
"""Cell 8: Test Embedded Models API"""
print("Embedded Models via FLUXModel")
print("=" * 60)

# List embedded models
embedded = model.list_embedded_models()
print(f"\nFound {len(embedded)} embedded models:")

for m in embedded:
    lazy = 'lazy' if m['lazy_load'] else 'eager'
    weights = '\u2713' if m['has_weights'] else '\u25cb'
    print(f"  {weights} {m['name']:15}: {m['base_model'][:40]} [{lazy}]")

# Test has_embedded_model
print("\nTesting has_embedded_model():")
test_models = ['instruct', 'vlm', 'voice', 'face', 'depth', 'whisper', 'nonexistent']
for name in test_models:
    has = model.has_embedded_model(name)
    status = '\u2713' if has else '\u2717'
    print(f"  {status} {name}")

In [ ]:
"""Cell 9: Memory/VRAM Estimation"""
print("Memory Estimation")
print("=" * 60)

def estimate_vram(params: int, dtype: str = 'fp16') -> float:
    """Estimate VRAM in GB for parameter count."""
    bytes_per_param = 2 if dtype == 'fp16' else 4 if dtype == 'fp32' else 1
    # Add ~20% overhead for activations
    return params * bytes_per_param * 1.2 / 1e9

print("\nEstimated VRAM Requirements (fp16):")
print(f"{'Component':<25} {'Params':>15} {'VRAM (GB)':>12}")
print("-" * 55)

# Native
for name, info in sorted(report.native_components.items(), key=lambda x: -x[1].parameters)[:5]:
    vram = estimate_vram(info.parameters)
    print(f"{name:<25} {info.parameters:>15,} {vram:>12.2f}")

# Embedded
if report.embedded_models:
    print()
    for name, info in sorted(report.embedded_models.items(), key=lambda x: -x[1].parameters):
        # Use actual quantization if known
        dtype = 'int4' if '4bit' in (info.quantization or '') else 'fp16'
        bytes_mult = 0.5 if dtype == 'int4' else 2
        vram = info.parameters * bytes_mult * 1.2 / 1e9
        print(f"{name:<25} {info.parameters:>15,} {vram:>12.2f}")

# Total
total_vram = estimate_vram(report.total_parameters)
print("-" * 55)
print(f"{'TOTAL (fp16 estimate)':<25} {report.total_parameters:>15,} {total_vram:>12.2f}")
print(f"\n\u26a0 Actual VRAM depends on quantization and lazy loading")

In [ ]:
"""Cell 10: Generate Markdown Report"""
from flux_inspector import generate_report
from IPython.display import Markdown, display

# Generate markdown report
md_report = generate_report(report, format='markdown')

# Display in notebook
display(Markdown(md_report))

# Optionally save to file
output_path = ROOT / 'output' / 'flux_apex_inspection.md'
output_path.parent.mkdir(exist_ok=True)
with open(output_path, 'w') as f:
    f.write(md_report)
print(f"\n\u2713 Report saved to: {output_path}")

In [ ]:
"""Cell 11: Component Tree View (Optional)"""
from flux_inspector import get_component_tree, print_tree

print("Component Tree (depth=2)")
print("=" * 60)

tree = get_component_tree(MODEL_PATH, max_depth=2)
print_tree(tree)

---

## Summary

This notebook provides:

1. **Dynamic inspection** — No hardcoded expectations
2. **Full component discovery** — Native, embedded, detection
3. **FLUXModel API testing** — Validates new methods
4. **Memory estimation** — VRAM requirements
5. **Exportable report** — Markdown format

Use this notebook to inspect any .flx file version.

**Next:** Run `flux_v6_integration.ipynb` for full integration testing.